# Phase 5.1 — Robustness and Stress Testing under 100,000 Flows Load

### 🎯 Objective
This notebook stress-tests the Quantum-RAG IDS v5.1 pipeline on an exponentially larger, completely unseen subset of the data (**100,000 network flows**) to verify system robustness, latency stability, and classification accuracy.

This tests the exact same quantum-inspired fusion logic used in Phase 4.1 against a fresh middle-slice of the data, isolated from the tail 8,400 slice used previously.


In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import chromadb
import time
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
import warnings
from collections import deque
import pickle

warnings.filterwarnings('ignore')

# ── Paths ──
NOTEBOOK_DIR = Path.cwd()
MAIN_FOLDER = NOTEBOOK_DIR.parent
DATA_DIR    = MAIN_FOLDER / "data" / "unified" / "ocean_v51"
CHROMA_DIR  = MAIN_FOLDER / "chromadb_store_v51"
ARTIFACTS   = MAIN_FOLDER / "artifacts"

print("✅ Libraries imported")


✅ Libraries imported


In [2]:
prep_path = ARTIFACTS / 'preprocessors_v51.pkl'
with open(prep_path, 'rb') as f:
    PP = pickle.load(f)

PP_BLOCK1     = PP['block1_scalers']
PP_BLOCK6     = PP['block6_scalers']
PP_QT         = PP['qt_byte_pkt']
SPORT_RARITY  = PP['sport_rarity_map']
DPORT_RARITY  = PP['dport_rarity_map']
DEFAULT_R     = PP.get('default_rarity', 1.0 / PP['total_rows_ocean'])


In [3]:
# ── Cell 3: vectorize_v51 + get_rag_context_batch (Phase 2.3 replicas) ─────────
#
# These replicas are bit-for-bit identical to Phase 2.3 — mandatory so that
# query vectors live in the same geometric space as the stored medoids.
#
# Phase 4.3 addition: Absolute Similarity Floor in get_rag_context_batch().
#   If top similarity < RAG_SIM_FLOOR (70%), override to UNKNOWN / 0.0.
#   Prevents low-confidence RAG matches from contaminating the accumulator.
#
# Updated for solo-style pipeline: returns `retrieval_results` list with ALL
# neighbours so that Evidence Accumulation (Phase 3.2) can group by attack_type.
# ─────────────────────────────────────────────────────────────────────────────

TOTAL_DIMS = 114
RAG_SIM_FLOOR = 70.0    # Phase 4.3 — similarity floor for RAG trust

PROTO_TOKENS     = ['tcp', 'udp', 'icmp', 'arp', 'ipv6', 'other']
SERVICE_TOKENS   = ['dns', 'http', 'ssl', 'ftp', 'ssh', 'smtp',
                    'dhcp', 'quic', 'ntp', 'rdp', 'pop3', 'other']
STATE_TOKENS     = ['PENDING', 'ESTABLISHED', 'REJECTED', 'RESET', 'OTHER']
PORT_FUNC_TOKENS = ['SCADA_CONTROL', 'IOT_MANAGEMENT', 'WEB_SERVICES',
                    'NETWORK_CORE',  'REMOTE_ACCESS',  'FUNC_EPHEMERAL', 'FUNC_UNKNOWN']
HTTP_METHOD_TOKENS = ['GET', 'POST', 'PUT', 'DELETE', 'HEAD', 'OPTIONS', 'PATCH', 'OTHER']
SSL_CIPHER_TOKENS  = [
    'TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256', 'TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384',
    'TLS_RSA_WITH_AES_128_GCM_SHA256',       'TLS_RSA_WITH_AES_256_GCM_SHA384',
    'TLS_RSA_WITH_AES_128_CBC_SHA',          'TLS_RSA_WITH_AES_256_CBC_SHA',
    'TLS_RSA_WITH_RC4_128_SHA',              'TLS_RSA_WITH_RC4_128_MD5',
    'TLS_RSA_WITH_3DES_EDE_CBC_SHA',         'TLS_DHE_RSA_WITH_AES_128_CBC_SHA',
    'TLS_ECDHE_ECDSA_WITH_AES_128_GCM_SHA256', 'other',
]
SCADA_PORTS    = frozenset({502, 102, 44818})
IOT_MGMT_PORTS = frozenset({1883, 5683, 8883})
WEB_PORTS      = frozenset({80, 443, 8080})
NET_CORE_PORTS = frozenset({53, 67, 68, 123})
REMOTE_PORTS   = frozenset({22, 23, 3389})
_PROTO_IDX   = {t: i for i, t in enumerate(PROTO_TOKENS)}
_SVC_IDX     = {t: i for i, t in enumerate(SERVICE_TOKENS)}
_STATE_IDX   = {t: i for i, t in enumerate(STATE_TOKENS)}
_METHOD_IDX  = {t: i for i, t in enumerate(HTTP_METHOD_TOKENS)}
_CIPHER_IDX  = {t: i for i, t in enumerate(SSL_CIPHER_TOKENS)}
_ABSENT_SVCS = frozenset({'<absent>', '-', 'unknown', '', 'none', '(empty)', 'nan'})
_DNS_QTYPE_MAP  = {1:0, 2:1, 5:2, 6:3, 12:4, 15:5, 16:6, 28:7, 33:8, 255:9}
_WEAK_SSL_VER   = frozenset({'sslv2','sslv3','tlsv1','tlsv10','tlsv1.0','tls1.0'})
_STRONG_SSL_VER = frozenset({'tlsv12','tlsv13','tlsv1.2','tlsv1.3','tls1.2','tls1.3'})

def _classify_port_vec(port_series):
    p = pd.to_numeric(port_series, errors='coerce').fillna(-1).astype(int)
    result = np.full(len(p), 6, dtype=np.int8)
    for _, func_idx, pset in [(0,0,SCADA_PORTS),(1,1,IOT_MGMT_PORTS),(2,2,WEB_PORTS),
                               (3,3,NET_CORE_PORTS),(4,4,REMOTE_PORTS)]:
        result[p.isin(pset).values] = func_idx
    result[(p.values > 49152) & (result == 6)] = 5
    return result

def vectorize_v51(df):
    """Map a DataFrame of v5.1-aligned ocean rows -> (N, 114) float32."""
    n = len(df); X = np.zeros((n, TOTAL_DIMS), dtype=np.float32); idx = df.index
    def _col(name, fill=0.0):
        return df[name].fillna(fill) if name in df.columns else pd.Series(fill, index=idx)
    def _str_col(name, fill=''):
        return (df[name].fillna(fill).astype(str).str.lower().str.strip()
                if name in df.columns else pd.Series(fill, index=idx))
    for col, mode, out_i in [('univ_duration','rs',0),('univ_bytes_in','qt',1),
                              ('univ_bytes_out','qt',2),('univ_pkts_in','qt',3),
                              ('univ_pkts_out','qt',4)]:
        vals = np.clip(_col(col,0.).values.astype(np.float64), 0., None)
        if mode=='qt' and col in qt_byte_pkt:
            X[:,out_i] = qt_byte_pkt[col].transform(vals.reshape(-1,1)).ravel().astype(np.float32)
        elif mode=='rs' and col in block1_scalers:
            X[:,out_i] = block1_scalers[col].transform(np.log1p(vals).reshape(-1,1)).ravel().astype(np.float32)
    proto = _str_col('raw_proto','other')
    X[np.arange(n), 5 + proto.map(lambda p: _PROTO_IDX.get(p, _PROTO_IDX['other'])).values] = 1.
    has_svc = _col('has_svc',0).values.astype(np.float32)
    svc = _str_col('raw_service','other')
    other_svc = _SVC_IDX['other']
    svc_idx = svc.map(lambda s: _SVC_IDX.get(s,other_svc) if s not in _ABSENT_SVCS else other_svc).values
    svc_ohe = np.zeros((n,12), dtype=np.float32); svc_ohe[np.arange(n), svc_idx] = 1.
    X[:,11:23] = svc_ohe * has_svc[:,np.newaxis]
    state = (df['raw_state_v51'].fillna('OTHER').astype(str).str.upper()
             if 'raw_state_v51' in df.columns else pd.Series('OTHER',index=idx))
    X[np.arange(n), 23 + state.map(lambda s: _STATE_IDX.get(s,_STATE_IDX['OTHER'])).values] = 1.
    X[np.arange(n), 28 + _classify_port_vec(_col('raw_sport',-1))] = 1.
    X[np.arange(n), 35 + _classify_port_vec(_col('raw_dport',-1))] = 1.
    DEFAULT_R = 1. / max(TOTAL_ROWS_OCEAN, 1)
    sport_str = _col('raw_sport',-1).values.astype(int).astype(str)
    dport_str = _col('raw_dport',-1).values.astype(int).astype(str)
    sr = np.array([sport_rarity.get(p, DEFAULT_R) for p in sport_str], dtype=np.float64)
    dr = np.array([dport_rarity.get(p, DEFAULT_R) for p in dport_str], dtype=np.float64)
    if pt_sport: X[:,42] = pt_sport.transform(sr.reshape(-1,1)).ravel().astype(np.float32)
    if pt_dport: X[:,43] = pt_dport.transform(dr.reshape(-1,1)).ravel().astype(np.float32)
    has_dns = _col('has_dns',0).values.astype(np.float32)
    qtype = _col('dns_qtype',-1).values.astype(int)
    qclass = _col('dns_qclass',-1).values.astype(int)
    rcode  = _col('dns_rcode',-1).values.astype(int)
    qt_arr = np.zeros((n,10), dtype=np.float32)
    for code, qi in _DNS_QTYPE_MAP.items(): qt_arr[qtype==code, qi] = 1.
    qt_arr[(qtype>0) & ~np.isin(qtype, list(_DNS_QTYPE_MAP.keys())), 9] = 1.
    X[:,44:54] = qt_arr * has_dns[:,np.newaxis]
    qc_arr = np.zeros((n,3), dtype=np.float32)
    qc_arr[qclass==1,0]=1.; qc_arr[qclass==3,1]=1.
    qc_arr[(qclass>=0)&(qclass!=1)&(qclass!=3),2]=1.
    X[:,54:57] = qc_arr * has_dns[:,np.newaxis]
    X[:,57] = ((rcode==0)&(has_dns>0)).astype(np.float32)
    X[:,58] = ((rcode>0)&(has_dns>0)).astype(np.float32)
    has_http = _col('has_http',0).values.astype(np.float32)
    http_m = (df['raw_http_method'].fillna('-').astype(str).str.strip().str.upper()
              if 'raw_http_method' in df.columns else pd.Series('-',index=idx))
    m_idx = http_m.map(lambda m: _METHOD_IDX.get(m,_METHOD_IDX['OTHER'])).values
    m_arr = np.zeros((n,8), dtype=np.float32)
    valid_m = (http_m!='-')&(http_m!='')&(http_m!='NAN')
    m_arr[valid_m.values, m_idx[valid_m.values]] = 1.
    X[:,59:67] = m_arr * has_http[:,np.newaxis]
    http_s = _col('http_status_code',-1).values.astype(int)
    s_arr = np.zeros((n,6), dtype=np.float32)
    s_arr[(http_s>=100)&(http_s<200),0]=1.; s_arr[(http_s>=200)&(http_s<300),1]=1.
    s_arr[(http_s>=300)&(http_s<400),2]=1.; s_arr[(http_s>=400)&(http_s<500),3]=1.
    s_arr[(http_s>=500)&(http_s<600),4]=1.; s_arr[http_s<0,5]=1.
    X[:,67:73] = s_arr * has_http[:,np.newaxis]
    req_b  = np.clip(_col('http_req_body_len',0).values.astype(np.float64),0,1e7)
    resp_b = np.clip(_col('http_resp_body_len',0).values.astype(np.float64),0,1e7)
    X[:,73] = (np.log1p(req_b)/np.log1p(1e7)).astype(np.float32)*has_http
    X[:,74] = (np.log1p(resp_b)/np.log1p(1e7)).astype(np.float32)*has_http
    X[:,75] = valid_m.values.astype(np.float32)*has_http
    X[:,76] = (http_s>=100).astype(np.float32)*has_http
    X[:,77] = (req_b>0).astype(np.float32)*has_http
    X[:,78] = (resp_b>0).astype(np.float32)*has_http
    has_ssl = _col('has_ssl',0).values.astype(np.float32)
    ssl_c = (df['raw_ssl_cipher'].fillna('').astype(str).str.strip()
             if 'raw_ssl_cipher' in df.columns else pd.Series('',index=idx))
    c_arr = np.zeros((n,12), dtype=np.float32)
    c_arr[np.arange(n), ssl_c.map(lambda c: _CIPHER_IDX.get(c,_CIPHER_IDX['other'])).values] = 1.
    X[:,80:92] = c_arr * has_ssl[:,np.newaxis]
    ssl_v = (df['raw_ssl_version'].fillna('').astype(str).str.strip().str.lower()
              .str.replace(' ','').str.replace('.','') if 'raw_ssl_version' in df.columns
              else pd.Series('',index=idx))
    X[:,92] = ssl_v.isin(_WEAK_SSL_VER).values.astype(np.float32)*has_ssl
    X[:,93] = ssl_v.isin(_STRONG_SSL_VER).values.astype(np.float32)*has_ssl
    X[:,94] = _col('ssl_established',0).values.astype(np.float32)*has_ssl
    has_unsw = _col('has_unsw',0).values.astype(np.float32)
    BLOCK6 = ['mom_mean','mom_stddev','mom_sum','mom_min','mom_max','mom_rate',
               'mom_srate','mom_drate','mom_TnBPSrcIP','mom_TnBPDstIP',
               'mom_TnP_PSrcIP','mom_TnP_PDstIP','mom_TnP_PerProto','mom_TnP_Per_Dport']
    for i, col in enumerate(BLOCK6):
        if col in block6_scalers:
            info = block6_scalers[col]; rs = info['scaler']; shift = info['shift']
            vals = _col(col,-1.).values.astype(np.float64); valid = vals != -1.
            out  = np.zeros(n, dtype=np.float32)
            if valid.any():
                out[valid] = rs.transform(np.log1p(vals[valid]+shift).reshape(-1,1)).ravel().astype(np.float32)
            X[:,95+i] = out * has_unsw
    X[:,109]=has_svc; X[:,110]=has_dns; X[:,111]=has_http; X[:,112]=has_ssl; X[:,113]=has_unsw
    np.nan_to_num(X, nan=0., posinf=0., neginf=0., copy=False)
    return X

def _l2_normalise(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0., 1., norms)
    return (X / norms).astype(np.float32)

def get_rag_context_batch(X_norm: np.ndarray, n_results: int = 3) -> list:
    """
    Query ChromaDB for a batch of pre-normalised vectors.

    Returns a list of dicts, each containing:
      - top_archetype, top_attack, top_similarity, top_distance (backward compat)
      - retrieval_results: list of ALL n_results neighbours with
        {similarity, attack_type, distance, rank} for Evidence Accumulation

    Phase 4.3 -- Absolute Similarity Floor:
      If the top neighbour's similarity_pct < RAG_SIM_FLOOR (70%),
      the packet is too far from any known medoid. Override to UNKNOWN
      and return empty retrieval_results so downstream accumulator
      treats it as pure uncertainty.
    """
    results = collection.query(
        query_embeddings=X_norm.tolist(),
        n_results=n_results,
        include=['metadatas', 'distances'],
    )
    outputs = []
    for i in range(len(X_norm)):
        neighbours = []
        for rank, (meta, dist) in enumerate(
            zip(results['metadatas'][i], results['distances'][i]), start=1
        ):
            dist_c  = max(0., float(dist))
            sim_pct = (1. - dist_c / 2.) * 100.
            neighbours.append({
                'rank'          : rank,
                'archetype'     : meta.get('ubt_archetype',       'UNKNOWN'),
                'attack_variant': meta.get('univ_specific_attack', 'UNKNOWN'),
                'dataset_source': meta.get('dataset_source',       'UNKNOWN'),
                'distance'      : round(dist_c, 6),
                'similarity_pct': round(sim_pct, 2),
            })
        top = neighbours[0] if neighbours else {}
        top_sim = top.get('similarity_pct', 0.)

        # ── Phase 4.3: Absolute Similarity Floor ──────────────────────────────
        if top_sim < RAG_SIM_FLOOR:
            outputs.append({
                'top_archetype'   : 'UNKNOWN',
                'top_attack'      : 'UNKNOWN',
                'top_similarity'  : 0.0,
                'top_distance'    : top.get('distance', 2.0),
                'retrieval_results': [],      # empty -> pure uncertainty downstream
            })
        else:
            # Build retrieval_results for Evidence Accumulation (Phase 3.2)
            # Each entry: similarity as fraction [0,1], attack_type = archetype
            retrieval_results = []
            for nb in neighbours:
                retrieval_results.append({
                    'similarity' : nb['similarity_pct'] / 100.0,
                    'attack_type': nb['archetype'],
                    'distance'   : nb['distance'],
                    'rank'       : nb['rank'],
                })
            outputs.append({
                'top_archetype'   : top.get('archetype',      'UNKNOWN'),
                'top_attack'      : top.get('attack_variant', 'UNKNOWN'),
                'top_similarity'  : top_sim,
                'top_distance'    : top.get('distance',       2.0),
                'retrieval_results': retrieval_results,
            })
    return outputs

print('vectorize_v51 and get_rag_context_batch defined \u2705')
print(f'  Output shape: (N, {TOTAL_DIMS}) float32  -- v5.1 schema lock')
print(f'  RAG similarity floor: {RAG_SIM_FLOOR}% (Phase 4.3)')
print(f'  RAG returns retrieval_results list for Evidence Accumulation')

vectorize_v51 and get_rag_context_batch defined ✅
  Output shape: (N, 114) float32  -- v5.1 schema lock
  RAG similarity floor: 70.0% (Phase 4.3)
  RAG returns retrieval_results list for Evidence Accumulation


In [4]:
ARCHETYPES = ['NORMAL', 'BOTNET_C2', 'SCAN', 'DOS_DDOS', 'BRUTE_FORCE', 'EXPLOIT', 'THEFT_EXFIL']
N_PER_ARCH = 14500

test_flows = []
print(f"⏳ Extracting {N_PER_ARCH} rows from the MIDDLE of each archetype...")

for arch in ARCHETYPES:
    arch_dir = DATA_DIR / f'ubt_archetype={arch}'
    if not arch_dir.exists(): continue
    files = sorted([f for f in arch_dir.glob("*.parquet")], key=lambda x: x.stat().st_size, reverse=True)
    if not files: continue
    
    mid_idx = len(files) // 2
    f = files[mid_idx]
    
    try:
        df_chunk = pq.read_table(f).to_pandas()
        start_row = len(df_chunk) // 2
        df_slice = df_chunk.iloc[start_row : start_row + N_PER_ARCH].copy()
        df_slice['ground_truth'] = arch
        test_flows.append(df_slice)
        print(f"  ✅ {arch:12s} : {len(df_slice):7,d} rows extracted")
    except Exception as e:
        print(f"  ❌ Failed reading {arch}: {e}")

df_test = pd.concat(test_flows, ignore_index=True)
print(f"\n🎯 Total Stress Test Flows: {len(df_test):,d}")


⏳ Extracting 14500 rows from the MIDDLE of each archetype...
  ✅ NORMAL       :      84 rows extracted
  ✅ BOTNET_C2    :  14,500 rows extracted
  ✅ SCAN         :  14,500 rows extracted
  ✅ DOS_DDOS     :  14,500 rows extracted
  ✅ BRUTE_FORCE  :  14,500 rows extracted
  ✅ EXPLOIT      :  14,500 rows extracted
  ✅ THEFT_EXFIL  :       2 rows extracted

🎯 Total Stress Test Flows: 72,586


In [5]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(name="ids_knowledge_base_v51")

def get_rag_context_batch(X_norm: np.ndarray, n_results=3, sim_floor=70.0):
    results = collection.query(query_embeddings=X_norm.tolist(), n_results=n_results, include=['metadatas', 'distances'])
    outputs = []
    for i in range(len(X_norm)):
        neighbors = []
        for rank, (meta, dist) in enumerate(zip(results['metadatas'][i], results['distances'][i])):
            dist_clamped = max(0.0, float(dist))
            sim_pct = (1.0 - dist_clamped / 2.0) * 100.0
            neighbors.append({'rank': rank, 'archetype': meta.get('archetype', 'UNKNOWN'), 'distance': dist_clamped, 'similarity': sim_pct})
        
        top_sim = neighbors[0]['similarity']
        if top_sim < sim_floor:
            outputs.append({'top_archetype': 'UNKNOWN', 'top_similarity': 0.0, 'results': []})
        else:
            outputs.append({'top_archetype': neighbors[0]['archetype'], 'top_similarity': top_sim, 'results': neighbors})
    return outputs


In [6]:
@dataclass
class FlowRecord:
    timestamp: float; vector: np.ndarray; top_similarity: float; top_archetype: str
    top_attack: str; packet_label: str; retrieval_results: list; flow_id: str

class StaticTimeWindow:
    def __init__(self, window_size_s=15.0, max_buffer=1000, benign_streak=3, benign_sim=85.0):
        self.window_size_s = window_size_s
        self.max_buffer = max_buffer
        self.benign_streak_threshold = benign_streak
        self.benign_sim_threshold = benign_sim
        self._buffer = deque(maxlen=max_buffer)
        self._benign_streak = 0
        self.slate_count = 0

    def ingest(self, flow: FlowRecord) -> list:
        if flow.top_archetype == 'NORMAL' and flow.top_similarity >= self.benign_sim_threshold:
            self._benign_streak += 1
        else:
            self._benign_streak = 0

        if self._benign_streak >= self.benign_streak_threshold:
            self._buffer.clear()
            self.slate_count += 1
            self._benign_streak = 0
            self._buffer.append(flow)
            return list(self._buffer)

        self._buffer.append(flow)
        cutoff = flow.timestamp - self.window_size_s
        while self._buffer and self._buffer[0].timestamp < cutoff:
            self._buffer.popleft()
        
        return list(self._buffer)

@dataclass
class AttackEvidence:
    attack_type: str; count: int; avg_similarity: float; max_similarity: float
    min_similarity: float; recurrence_score: float; flow_ids: list

def accumulate_evidence(window_flows: list) -> dict:
    if not window_flows: return {}
    n_flows = len(window_flows)
    data = {}
    f_set = {}
    
    for f in window_flows:
        for r in f.retrieval_results:
            atype = r.get('archetype', 'UNKNOWN')
            sim = r.get('similarity', 0.0) / 100.0  # scale 0 to 1
            if atype not in data:
                data[atype] = {'sims': [], 'fids': []}
                f_set[atype] = set()
            data[atype]['sims'].append(sim)
            data[atype]['fids'].append(f.flow_id)
            f_set[atype].add(id(f))
            
    evidence = {}
    for atype, d in data.items():
        sims = d['sims']
        evidence[atype] = AttackEvidence(
            attack_type=atype, count=len(sims),
            avg_similarity=sum(sims)/len(sims),
            max_similarity=max(sims), min_similarity=min(sims),
            recurrence_score=len(f_set[atype])/n_flows,
            flow_ids=d['fids']
        )
    return evidence

def quantum_fuse(evidence_dict: dict):
    if not evidence_dict:
        return {'top_attack': 'UNKNOWN', 'top_prob': 0.0, 'entropy_ratio': 1.0, 'probs': {}}
        
    amplitudes = {}
    for atype, ev in evidence_dict.items():
        phase = np.exp(2.0 * ev.avg_similarity)
        amp = np.sqrt(ev.count * phase * (ev.recurrence_score + 0.1))
        amplitudes[atype] = amp
        
    sum_sq = sum(a**2 for a in amplitudes.values())
    probs = {at: (a**2)/sum_sq for at, a in amplitudes.items()} if sum_sq > 1e-15 else {}
    
    entropy_ratio = 1.0
    if len(probs) > 1:
        h = -sum(p * np.log2(p) for p in probs.values() if p > 1e-15)
        entropy_ratio = h / np.log2(max(len(probs), 2))
    elif len(probs) == 1:
        entropy_ratio = 0.0
        
    top_attack, top_prob = max(probs.items(), key=lambda x: x[1]) if probs else ('UNKNOWN', 0.0)
    
    # Squeeze
    if top_attack != 'NORMAL':
        gamma = 0.70 if entropy_ratio > 0.80 else 0.85
        squeezed = 1.0 - (1.0 - top_prob)**gamma
        top_prob = max(top_prob, 0.6 * top_prob + 0.4 * squeezed)
        probs[top_attack] = top_prob # Note: invalidates perfect sum to 1, but used for verdict
        
    return {'top_attack': top_attack, 'top_prob': top_prob, 'entropy_ratio': entropy_ratio, 'probs': probs}

def generate_ids_verdict(qf_result: dict, top_sim: float):
    if top_sim < 82.0: return 'CLEAR'
    aprob = qf_result['top_prob']
    eratio = qf_result['entropy_ratio']
    
    if aprob >= 0.95 and top_sim > 90.0: return 'CRITICAL'
    if aprob >= 0.85 and top_sim > 85.0: return 'WARNING'
    if eratio > 0.85: return 'MONITOR'
    return 'ELEVATED'


In [7]:
# =============================================================================
# Run Pipeline
# =============================================================================
BATCH_SIZE = 1000
total_batches = int(np.ceil(len(df_test) / BATCH_SIZE))

predictions = []
ground_truths = df_test['ground_truth'].tolist()
BASE_TS = 1700000000.0

print(f"🚀 Launching Stress Test on {len(df_test):,} flows over {total_batches} batches...")

start_time = time.time()
time_window = StaticTimeWindow(window_size_s=15.0)

for b in tqdm(range(total_batches)):
    batch_df = df_test.iloc[b * BATCH_SIZE : (b+1) * BATCH_SIZE]
    X = vectorize_v51(batch_df)
    X_norm = normalize_vectors(X)
    
    rag_outs = get_rag_context_batch(X_norm, n_results=3, sim_floor=70.0)
    
    for relative_i, out in enumerate(rag_outs):
        i = (b * BATCH_SIZE) + relative_i
        
        flow = FlowRecord(
            timestamp=BASE_TS + (i * 0.1),
            vector=X_norm[relative_i],
            top_similarity=out['top_similarity'],
            top_archetype=out['top_archetype'],
            top_attack=out.get('results', [{'attack_variant': 'unknown'}])[0].get('attack_variant', 'unknown') if out['results'] else 'unknown',
            packet_label=ground_truths[i],
            retrieval_results=out['results'],
            flow_id=f"flow_{i}"
        )
        
        buffer = time_window.ingest(flow)
        ev = accumulate_evidence(buffer)
        qf = quantum_fuse(ev)
        verdict = generate_ids_verdict(qf, out['top_similarity'])
        
        # Binary to attack multiclass for simple evaluation
        prediction_arch = qf['top_attack']
        if verdict in ['CLEAR', 'MONITOR', 'ELEVATED'] and prediction_arch != 'NORMAL':
            # Soft suppression
            if verdict == 'CLEAR': prediction_arch = 'NORMAL'
            
        predictions.append(prediction_arch)

total_time = time.time() - start_time
pps = len(df_test) / total_time
latency = (total_time / len(df_test)) * 1e6

print(f"\n✅ Stress Test Completed")
print(f"⏱️ Total Time: {total_time:.2f} s")
print(f"🚀 Throughput: {pps:.0f} pps")
print(f"⚡ Latency   : {latency:.2f} µs/packet")


🚀 Launching Stress Test on 72,586 flows over 73 batches...


  0%|          | 0/73 [00:00<?, ?it/s]

NameError: name 'block1_scalers' is not defined

In [ ]:
# =============================================================================
# Evaluation Metrics
# =============================================================================
from sklearn.metrics import classification_report

y_true = ground_truths[:len(predictions)]
y_pred = predictions

print(classification_report(y_true, y_pred, zero_division=0))
